# RAG with Ollama, LangChain, and PDFs

This notebook demonstrates Retrieval-Augmented Generation (RAG) using:
- **Ollama**: Local LLM model
- **LangChain**: Framework for building RAG applications
- **PyPDFLoader**: Load PDF documents
- **Vector Store**: Store document embeddings for semantic search
- **RetrievalQA**: Question-answering chain with document retrieval

## Section 1: Import Required Libraries

In [ ]:
# Import Required Libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.callbacks import StreamingStdOutCallbackHandler, CallbackManager
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

## Section 2: Load PDF Documents with PyPDFLoader

Specify the path to your PDF file and load it using PyPDFLoader.

In [ ]:
# Load PDF Documents
pdf_path = "sample.pdf"  # Change this to your PDF file path

# Check if file exists
if os.path.exists(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from the PDF")
    print(f"First page preview: {documents[0].page_content[:200]}...")
else:
    print(f"PDF file not found at: {pdf_path}")
    print("Please ensure you have a PDF file and update the pdf_path variable")

## Section 3: Split Documents with LangChain Text Splitter

Apply RecursiveCharacterTextSplitter with chunk_size=400 and chunk_overlap=50

In [ ]:
# Split Documents with Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,          # Size of each chunk
    chunk_overlap=50,        # Overlap between chunks
    separators=["\n\n", "\n", " ", ""]  # Separators to use for splitting
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")
print(f"First chunk:\n{chunks[0].page_content}\n")
print(f"Chunk metadata: {chunks[0].metadata}")

## Section 4: Initialize Ollama Model Connection

Set up connection to a local Ollama instance. Ensure Ollama is running with the desired model.

In [ ]:
# Initialize Ollama Model Connection
# Make sure Ollama is running: ollama serve

# Set up callback handler for streaming output
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

# Initialize Ollama LLM (ChatOllama for chat models)
# Change "mistral" to your desired model (e.g., "llama2", "neural-chat", "dolphin-mixtral")
llm = ChatOllama(
    model="llama3.2:1b",
    base_url="http://localhost:11434",  # Default Ollama API endpoint
    callbacks=[StreamingStdOutCallbackHandler()],
    verbose=True
)

print("Ollama model initialized successfully!")
print(f"Model: LLAMA 3.2")
print(f"Base URL: http://localhost:11434")

## Section 5: Create Vector Store with FAISS

Create embeddings for document chunks and store them using FAISS (Facebook AI Similarity Search) for efficient semantic similarity search.

In [ ]:
# Create Vector Store with Ollama Embeddings using FAISS
print("Creating embeddings and vector store... This may take a moment...")

# Initialize Ollama embeddings
embeddings = OllamaEmbeddings(
    base_url="http://localhost:11434",
    model="all-minilm"
)



Creating embeddings and vector store... This may take a moment...


ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

In [12]:
# Create FAISS vector store from documents
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Save the FAISS index to disk for persistence
vector_store.save_local("./faiss_db")

print(f"Vector store created successfully!")
print(f"Total vectors stored: {len(chunks)}")
print(f"FAISS index saved at: ./faiss_db")

Vector store created successfully!
Total vectors stored: 2994
FAISS index saved at: ./faiss_db


## Section 6: Set Up RetrievalQA Chain

Configure RetrievalQA with the Ollama model, vector store as retriever, and appropriate chain type.

In [13]:
# Set Up Retrieval Chain (Modern LCEL Approach)
# Create a retriever from the vector store
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Retrieve top 3 most similar documents
)

# Create a prompt template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""Answer the question based on the following context:

Context: {context}

Question: {question}

Answer:""")

# Create the RAG chain using LCEL
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Retrieval chain created successfully!")
print("Chain type: LCEL (LangChain Expression Language)")
print("Retriever search results: top 3 documents")

Retrieval chain created successfully!
Chain type: LCEL (LangChain Expression Language)
Retriever search results: top 3 documents


## Section 7: Query the RAG System

Execute queries against the RAG system and display results with retrieved context.

In [14]:
# Query the RAG System
# Example queries
queries = [
    "What is the main topic of this document?",
    "Can you summarize the key points?",
    "What specific information is covered?"
]

# Run queries
for query in queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}\n")
    
    # Get the answer
    answer = qa_chain.invoke(query)
    print(f"Answer:\n{answer}\n")
    
    # Get source documents separately
    source_docs = retriever.invoke(query)
    print("Source Documents:")
    for i, doc in enumerate(source_docs, 1):
        print(f"\n  Document {i}:")
        print(f"  Source: {doc.metadata}")
        print(f"  Content: {doc.page_content[:200]}...")


Query: What is the main topic of this document?



ResponseError: model requires more system memory (15.9 GiB) than is available (14.1 GiB) (status code: 500)

## Additional Notes and Configuration

### Prerequisites
- **Ollama**: Install from [ollama.ai](https://ollama.ai)
- **LangChain**: `pip install langchain`
- **PyPDF**: `pip install pypdf`
- **FAISS**: `pip install faiss-cpu` (or `faiss-gpu` for GPU support)

### Running Ollama
```bash
ollama serve
```
In another terminal, pull a model:
```bash
ollama pull mistral
```

### Customization Options
- **Chunk Size**: Increase to 800-1000 for larger documents, decrease to 200 for more granular chunks
- **Chunk Overlap**: Typically 10-20% of chunk size for context preservation
- **LLM Models**: Try "llama2", "neural-chat", "dolphin-mixtral", "openhermes"
- **Chain Type**: Use "refine" for longer documents, "map_reduce" for better accuracy with large datasets
- **Search K**: Increase to retrieve more context (e.g., k=5 or k=10)

### Performance Tips
- FAISS is optimized for fast similarity search with CPU or GPU acceleration
- Use GPU support (`faiss-gpu`) for better performance with large datasets
- FAISS indices are saved locally in `./faiss_db` directory for persistence
- Load existing index: `FAISS.load_local("./faiss_db", embeddings)`